PART A

In [6]:
# 1.
import pandas as pd

df = pd.read_csv("periodic_table.csv")

df.head()

# 2.
mass_dict = pd.Series(df.AtomicMass.values, index=df.Symbol).to_dict()
print(mass_dict["H"])   # should give ~1.008
print(mass_dict["O"])   # should give ~15.999


# 3.
import re

def molecular_mass(formula, mass_dict):
    tokens = re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    
    total_mass = 0
    for (element, count) in tokens:
        count = int(count) if count else 1
        total_mass += mass_dict[element] * count
    return total_mass

print(molecular_mass("H2O", mass_dict))       # ~18.015
print(molecular_mass("C6H12O6", mass_dict))  # ~180.156

# 4. 

def parse_formula(formula, mass_dict):
    while "(" in formula:
        sub = re.search(r'\(([^()]+)\)(\d*)', formula)
        if not sub:
            break
        inside, mult = sub.groups()
        mult = int(mult) if mult else 1
        expanded = "".join([f"{el}{int(num)*mult if num else mult}" 
                            for el, num in re.findall(r'([A-Z][a-z]?)(\d*)', inside)])
        formula = formula[:sub.start()] + expanded + formula[sub.end():]
    return molecular_mass(formula, mass_dict)

print(parse_formula("Ca(OH)2", mass_dict))   # ~74.093

# 5.
def hydrated_mass(formula, mass_dict):
    parts = formula.split(".")
    total = 0
    for part in parts:
        # If part starts with a number, separate multiplier
        m = re.match(r'(\d+)(.*)', part)
        if m:
            mult, rest = m.groups()
            total += int(mult) * parse_formula(rest, mass_dict)
        else:
            total += parse_formula(part, mass_dict)
    return total

print(hydrated_mass("CuSO4.5H2O", mass_dict))   # ~249.685

1.008
15.999
18.015
180.156
74.09400000000001
249.69100000000003


PART B

In [9]:
# 1. 
import sympy as sp
import re

def parse_formula_count(formula):
    while "(" in formula:
        sub = re.search(r'\(([^()]+)\)(\d*)', formula)
        inside, mult = sub.groups()
        mult = int(mult) if mult else 1
        expanded = "".join([f"{el}{int(num)*mult if num else mult}" 
                            for el,num in re.findall(r'([A-Z][a-z]?)(\d*)', inside)])
        formula = formula[:sub.start()] + expanded + formula[sub.end():]
    counts = {}
    for el,num in re.findall(r'([A-Z][a-z]?)(\d*)', formula):
        counts[el] = counts.get(el,0) + (int(num) if num else 1)
    return counts


def balance_reaction(reactants, products):
    species = reactants + products
    n_r = len(reactants)
    elems = sorted({e for f in species for e in parse_formula_count(f)})

    # Build atom balance matrix (reactants +, products -)
    A = sp.Matrix([
        [parse_formula_count(f).get(e,0) * (1 if j < n_r else -1)
         for j,f in enumerate(species)]
        for e in elems
    ])

    ns = A.nullspace()
    if not ns:
        raise ValueError("No non-trivial solution found.")
    vec = ns[0]

    # scale to integers
    denoms = [c.as_numer_denom()[1] for c in vec]
    lcm = sp.ilcm(*denoms)
    coeffs = [int(c*lcm) for c in vec]

    # make positive
    if any(c < 0 for c in coeffs):
        coeffs = [-c for c in coeffs]

    # reduce by gcd
    from math import gcd
    g = gcd(*(c for c in coeffs if c != 0))
    coeffs = [c//g for c in coeffs]

    return coeffs


print(balance_reaction(["H2", "O2"], ["H2O"]))  
# Expected: [2,1,2]

print(balance_reaction(["C2H6","O2"], ["CO2","H2O"]))  
# Expected: [2,7,4,6]

print(balance_reaction(["Fe","O2"], ["Fe2O3"]))  
# Expected: [4,3,2]


# 2.
def mass_conserved(reactants, products, coeffs, mass_dict):
    n = len(reactants)
    rm = sum(hydrated_mass(r, mass_dict)*coeffs[i] for i,r in enumerate(reactants))
    pm = sum(hydrated_mass(p, mass_dict)*coeffs[n+i] for i,p in enumerate(products))
    return abs(rm - pm) < 1e-6


coeffs = balance_reaction(["H2", "O2"], ["H2O"])
print(mass_conserved(["H2", "O2"], ["H2O"], coeffs, mass_dict))  # True


[2, 1, 2]
[2, 7, 4, 6]
[4, 3, 2]
True
